# Plot PCC/RMSE regional forecast skill

Plots cached lead-window skill products; it does not reread raw model data.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")

groups = ("Jan2012", "Jun2012")
run = "fc"
variables = ("UBOT", "VBOT", "PSL", "U850", "V850", "T850", "Q850", "Z500", "PRECT", "FLUT")
regions = ("GLOBAL", "GLOBAL_LAND", "NINO3.4", "MAR_CONT", "CONUS", "NH_POLAR")
metric = "ACC"
vmin, vmax, nlev = 0.0, 1.0, 11
show = True

In [ ]:
from pathlib import Path
import sys

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from configs.s2s_config import EXPERIMENT_GROUPS, OBS_VARIABLES, PCC_REGIONS, TCC_REGIONS, LEAD_WINDOWS, WEEKLY_WINDOWS, AVAILABLE_PCC_METRICS, AVAILABLE_PCC_MAP_METRICS, AVAILABLE_TCC_METRICS
from util.s2s_experiments import build_experiments
from util.s2s_observations import OBS_REGISTRY, get_obs_file


## Notes: available selections

- Experiment groups: `Jan2012`, `Jun2012`.
- Runs: `fc` (standard forecast) or `wc` (water-cycle forecast).
- Observations and variables: `ERA5` supports UBOT, VBOT, PRECT, PSL, T925, Q925, U850, V850, T850, Q850, Z500, OMEGA850/500/200, U/V500, U/V200, FLDS/C, FLNS/C, FSDS/C, FSNS/C, LHFLX, SHFLX, QFLX, TREFHT, TS, PRECSL, PRECC, and PRECL; `GPCP` and `GPM` support PRECT; `NOAA-OLR` supports FLUT.
- Regions: GLOBAL, land/ocean where applicable, NINO3.4, MAR_CONT, CONUS, NH_POLAR.
- Existing NetCDF diagnostics are reused. Set `force_compute = True` to rebuild selected products.

Available metrics: `ACC`, `RMSE`, `ACC_member_mean`, `ACC_member_quantile`, `RMSE_member_mean`, `RMSE_member_quantile`.

Exact regional fields: `ACC`, `RMSE`; with members, `ACC_member`, `RMSE_member`, `ACC_member_mean`, `RMSE_member_mean`, `ACC_member_std`, `RMSE_member_std`, `ACC_member_quantile`, `RMSE_member_quantile`.


In [ ]:
from util.s2s_pcc_plotting import S2SSkillPlotConfig, S2SSkillPlotter

data_dir, figure_dir = workflow_output_dirs("fcst_s2s_pcc", output_root=DIAGNOSTIC_OUTPUT_ROOT)
exp_order = tuple(exp for group in groups for exp in EXPERIMENT_GROUPS[group]["models"])
obs_by_var = {"PRECT":"GPM", "FLUT":"NOAA-OLR"}
plotter = S2SSkillPlotter(S2SSkillPlotConfig(skill_dir=data_dir, file_glob=f"s2s_skill_*_daily_{run}_*_*.nc", default_obs="ERA5", obs_by_var=obs_by_var, vars_include=variables, exp_order=exp_order, regions_include=regions, out_dir=figure_dir, tag=f"{'_'.join(groups)}_{run}"))
plotter.plot_quadrant_heatmap_all_regions(regions=regions, metric=metric, vmin=vmin, vmax=vmax, nlev=nlev, show=show)
